# IEEE IoT-J R1 — One-run evidence bundle (Kaggle)

**Scope:** deterministic source-host-disjoint binary Edge-IIoTset evaluation, validation-selected thresholds, capture-feature ablation (same locked split), prediction alignment, capture confounding, time provenance gate, cross-dataset feature-eligibility gate, and compute-platform latency proxy.

**Scientific constraints:** Source IP is not a verified physical-device ID. Kaggle timing is not Raspberry Pi latency. No cross-dataset KD experiment may be claimed without an independently verified mapping of features and units. Timestamp anomalies mean chronological validity is not assumed. Fixed host split and all student comparisons use identical partitions. Test labels never select the operating point.

This notebook reruns full baseline training and feature ablation, five seeds each; computationally demanding on Kaggle. Outputs are persisted after each completed seed.

In [ ]:
import os, re, json, time, pathlib, gc, warnings, random
from collections import defaultdict
import numpy as np, pandas as pd
from scipy.special import softmax
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer
from sklearn.pipeline import Pipeline
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings('ignore',category=FutureWarning)
INPUT=pathlib.Path(os.getenv('KAGGLE_INPUT_ROOT','/kaggle/input'))
ROOT=pathlib.Path('/kaggle/working/kd_r1_all_in_one' if pathlib.Path('/kaggle/working').exists() else './kd_r1_all_in_one');ROOT.mkdir(parents=True,exist_ok=True)
SEEDS=[13,42,77,2024,31337]; DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
FAST_DEV_RUN=False; EPOCHS=4 if FAST_DEV_RUN else 12
MAX_ROWS_PER_SOURCE_FILE=12000; CHUNKSIZE=25000
MIN_HOSTS=5; MIN_TEST_HOSTS=1; MIN_PER_LABEL_EACH=20; MIN_TEST_LABEL_COVERAGE=0.80
TIME_PARSE_RATE_MIN=.99
# Time qualification must be explicitly reviewed against dataset provenance before enabling.
CAPTURE_TIME_PROVENANCE_APPROVED=False
print('Kaggle input',INPUT,'device',DEVICE,'output',ROOT)

MAX_TRAIN_ROWS=50000  # fixed budget sampled only after splitting, seed independent
MIN_BINARY_PER_SPLIT=100
MAX_BINARY_PREVALENCE_GAP=0.45  # warning threshold, not an automatic success claim


## Dataset discovery — raw captures only, no hardcoded dataset slug

In [ ]:
def discover_raw(root):
    if not root.exists():raise FileNotFoundError('Attach Edge-IIoTset dataset to Kaggle using Add Input')
    found=[]
    for p in root.rglob('*.csv'):
        s=str(p).lower()
        if not ('edgeiiot' in s or 'edge-iiot' in s or 'iiotset' in s):continue
        if 'dnn-edgeiiot' in p.name.lower() or 'selected dataset for ml' in s:continue
        try: cols=list(pd.read_csv(p,nrows=0).columns)
        except Exception:continue
        low={c.lower().strip():c for c in cols}
        if 'ip.src_host' in low and 'frame.time' in low and ('attack_type' in low or 'attack_label' in low):
            found.append((p,cols))
    return sorted(found,key=lambda z:str(z[0]))
FILES=discover_raw(INPUT)
print('Raw capture CSVs:',len(FILES));print('Examples:',*[str(p) for p,_ in FILES[:4]],sep='\n')
if not FILES:raise RuntimeError('No raw capture files exposing ip.src_host and frame.time were found; do not substitute reduced DNN export')
schema=pd.DataFrame([{'file':str(p),'columns':len(cols),'has_host':'ip.src_host' in cols,'has_time':'frame.time' in cols,
                       'has_attack_type':'Attack_type' in cols} for p,cols in FILES]);schema.to_csv(ROOT/'raw_capture_inventory.csv',index=False)


## Audited, bounded per-capture sampling and binary target construction

In [ ]:
BLOCKED={'ip.src_host','ip.dst_host','frame.time','attack_label','attack_type','label','class','tcp.srcport','tcp.dstport',
         'udp.port','http.tls_port','arp.src.proto_ipv4','arp.dst.proto_ipv4','icmp.transmit_timestamp'}
BLOCK_TERMS=('payload','http.file_data','http.request.uri','dns.qry.name','mqtt.msg','mqtt.topic','mac.src','mac.dst','tcp.options')
META=['ip.src_host','frame.time','Attack_type','Attack_label']
def numeric_candidates(cols, p):
    candidates=[x for x in cols if x.strip().lower() not in BLOCKED and not any(k in x.lower() for k in BLOCK_TERMS)]
    head=pd.read_csv(p,usecols=[x for x in candidates if x in cols],nrows=200,low_memory=False)
    return [c for c in head if pd.to_numeric(head[c],errors='coerce').notna().mean()>=.90]
# Resolve common feature set before reading large files; do not mix columns by position.
COMMON=set.intersection(*(set(numeric_candidates(cols,p)) for p,cols in FILES))
FEATURES=sorted(COMMON)
print('Shared eligible numeric predictors:',len(FEATURES),FEATURES[:15]);assert len(FEATURES)>=5
rows=[]; diagnostics=[]
for fi,(p,cols) in enumerate(FILES):
    rand=np.random.default_rng(1729+fi); pools=[]; seen=0
    use=list(dict.fromkeys(FEATURES+[m for m in META if m in cols]))
    for ch in pd.read_csv(p,usecols=use,chunksize=CHUNKSIZE,low_memory=False):
        ch=ch.copy();seen+=len(ch)
        if 'Attack_type' not in ch:continue
        ch['Attack_type']=ch['Attack_type'].astype(str).str.strip()
        ch=ch[ch['Attack_type'].notna() & ~ch['Attack_type'].str.lower().isin(['nan',''])]
        if len(ch)==0:continue
        # Random sampling independent of row ordering; file-specific bounded pool.
        quota=max(1,int(np.ceil(MAX_ROWS_PER_SOURCE_FILE*len(ch)/max(seen,1))))
        pools.append(ch.sample(n=min(len(ch),quota),random_state=int(rand.integers(1,2**31-1))))
        if len(pools)>20:
            merged=pd.concat(pools,ignore_index=True)
            pools=[merged.sample(n=min(len(merged),MAX_ROWS_PER_SOURCE_FILE),random_state=1729+fi)]
    if pools:
        part=pd.concat(pools,ignore_index=True)
        if len(part)>MAX_ROWS_PER_SOURCE_FILE:part=part.sample(n=MAX_ROWS_PER_SOURCE_FILE,random_state=1729+fi)
        part['__capture_file__']=str(p.relative_to(INPUT));rows.append(part)
        diagnostics.append({'capture_file':str(p),'scanned':seen,'retained':len(part),'labels':part.Attack_type.nunique()})
    print(p.name,'scanned',seen,'retained',len(part) if pools else 0)
if not rows:raise RuntimeError('No labeled raw capture rows')
RAW=pd.concat(rows,ignore_index=True)
RAW['__host__']=RAW['ip.src_host'].astype('string').str.strip().str.lower().replace({'':pd.NA,'nan':pd.NA,'none':pd.NA})
RAW['__label__']=RAW['Attack_type'].astype('string').str.strip()
RAW['__time__']=pd.to_datetime(RAW['frame.time'],errors='coerce',utc=True)
RAW['__capture_file__']=RAW['__capture_file__'].astype('string')
RAW['__row_id__']=np.arange(len(RAW))
pd.DataFrame(diagnostics).to_csv(ROOT/'sampling_manifest.csv',index=False)
print('Retained',len(RAW),'classes',RAW.__label__.nunique(),'hosts',RAW.__host__.nunique(),'time parse',RAW.__time__.notna().mean())

# Target is formed from raw Attack_type. No capture filename is used for classification.
RAW['__binary__']=np.where(RAW['__label__'].str.casefold().eq('normal'),'benign','attack')
assert set(RAW['__binary__'])=={'benign','attack'}, 'Both binary classes required'
RAW[['__host__','__label__','__binary__']].groupby(['__label__','__binary__']).size().to_csv(ROOT/'binary_label_map_counts.csv')
print('Binary classes:',RAW['__binary__'].value_counts().to_dict())


## Metadata and confounding audit

In [ ]:
hist=pd.crosstab(RAW['__host__'].fillna('MISSING'),RAW['__label__'])
hist.to_csv(ROOT/'host_by_attack_counts.csv')
per_class=(hist>0).sum(axis=0).rename('host_count').reset_index().rename(columns={'__label__':'attack_label'})
per_class.to_csv(ROOT/'attack_host_support.csv',index=False)
file_time=RAW.groupby('__capture_file__',observed=True).agg(n=('__row_id__','size'),
                   first_time=('__time__','min'),last_time=('__time__','max'),hosts=('__host__','nunique'),
                   labels=('__label__','nunique')).reset_index()
file_time.to_csv(ROOT/'capture_time_audit.csv',index=False)
print('Hosts per attack class:',per_class.to_string(index=False));print('Capture spans:',file_time.head(20).to_string(index=False))
missing_host=float(RAW.__host__.isna().mean());missing_time=float(RAW.__time__.isna().mean())
labels=sorted(RAW.__label__.dropna().unique().tolist());counts=RAW.__label__.value_counts()
print('Missing host fraction',missing_host,'missing time fraction',missing_time)

ct2=pd.crosstab(RAW['__host__'].fillna('MISSING'),RAW['__binary__'])
ct2.to_csv(ROOT/'binary_host_contingency.csv')
print('Hosts carrying both benign and attacks:',int(((ct2>0).all(axis=1)).sum()))


## Locked split: binary class coverage, complete host separation, pre-specified design

A deterministic search explores candidate splits using **only source-host groups, counts, and binary labels**. It never evaluates model predictions or tunes on test-set performance. Candidates must contain enough benign and attack samples in every partition. Source-host independence is a *proxy* and does not establish identity of physical devices. Strong host/class confounding is reported, not concealed.

In [ ]:
import hashlib
from sklearn.model_selection import GroupShuffleSplit
SPLIT_SEED=314159
TARGET_FRACTIONS={'train':0.70,'val':0.10,'test':0.20}
SPLIT_CANDIDATES=100

def split_binary_hosts(frame,seed=SPLIT_SEED,candidates=SPLIT_CANDIDATES):
    f=frame.loc[frame['__host__'].notna() & frame['__binary__'].isin(['benign','attack'])].copy().reset_index(drop=True)
    hist=pd.crosstab(f['__host__'].astype(str),f['__binary__']).reindex(columns=['benign','attack'],fill_value=0)
    hosts=hist.index.to_numpy(dtype=object); counts=hist.to_numpy(dtype=np.int64)
    totals=counts.sum(axis=1); total=totals.sum(); overall=counts.sum(axis=0)/total
    report={'host_count':len(hosts),'sample_count':len(f),'class_counts':dict(zip(['benign','attack'],map(int,counts.sum(axis=0)))),'seed':seed,'candidates':candidates, 'min_binary_per_split':MIN_BINARY_PER_SPLIT}
    if len(hosts)<9 or (counts>0).sum(axis=0).min()<3:
        report.update(status='INELIGIBLE',reason='Too few independent hosts for three-way coverage');return f,None,report
    rng=np.random.default_rng(seed)
    # Whole hosts assigned, not individual rows. No model metrics consulted.
    best=None; feasible_count=0
    for attempt in range(candidates):
        group=np.full(len(hosts),-1,dtype=np.int8)
        perm=rng.permutation(len(hosts))
        # Randomized greedy group allocation with class-deficit and total-size balancing
        # anchors benign-hosts in each fold to improve feasibility under label/host confounding
        benign_hosts=np.where(counts[:,0]>0)[0]
        if len(benign_hosts)>=3:
            anchors=benign_hosts[rng.permutation(len(benign_hosts))[:3]]
            group[anchors]=np.array([0,1,2],dtype=np.int8)
        order=perm[np.argsort(-totals[perm],kind='stable')]
        group_counts=np.stack([counts[group==k].sum(axis=0) for k in range(3)])
        for j in order:
            if group[j]>=0:continue
            candidate_scores=[]
            for k in range(3):
                next_counts=group_counts.copy();next_counts[k]+=counts[j]
                current_size=next_counts.sum(axis=1)
                ratios=current_size/total
                size_loss=np.sum(np.abs(ratios-np.array([.70,.10,.20])))
                class_targets=overall[None,:]*np.array([.70,.10,.20])[:,None]*total
                class_loss=np.sum(np.abs(next_counts-class_targets)/np.maximum(class_targets,1))
                coverage_penalty=np.sum((next_counts[:,0]==0))*0.2
                candidate_scores.append(size_loss + 0.15*class_loss + coverage_penalty)
            k=int(np.argmin(candidate_scores)); group[j]=k;group_counts[k]+=counts[j]
        valid=bool((group_counts>=MIN_BINARY_PER_SPLIT).all())
        proportions=group_counts.sum(axis=1)/total
        class_prevalence=group_counts[:,1]/np.maximum(group_counts.sum(axis=1),1)
        # Selection uses only fixed balance objectives, never downstream model outcomes.
        objective=float(np.abs(proportions-[.70,.10,.20]).sum()+0.3*np.abs(class_prevalence-overall[1]).sum())
        if valid:
            feasible_count+=1
            if best is None or objective<best[0]:best=(objective,group.copy(),group_counts.copy(),attempt)
    report['feasible_candidates']=feasible_count
    if best is None:
        report.update(status='INELIGIBLE',reason='No candidate meets minimum binary coverage');return f,None,report
    _,allocation,grp_counts,attempt=best
    assigned=dict(zip(hosts,allocation))
    f['__split__']=f['__host__'].map(assigned).map({0:'train',1:'val',2:'test'})
    assert not f['__split__'].isna().any()
    sets={role:set(f.loc[f['__split__']==role,'__host__']) for role in ('train','val','test')}
    assert not (sets['train']&sets['val'] or sets['train']&sets['test'] or sets['val']&sets['test'])
    audit_counts=pd.crosstab(f['__split__'],f['__binary__']).reindex(index=['train','val','test'],columns=['benign','attack'],fill_value=0)
    assert (audit_counts>=MIN_BINARY_PER_SPLIT).all().all()
    roles=[assigned[h] for h in sorted(assigned)]
    fingerprint=hashlib.sha256(json.dumps(list(zip(sorted(assigned),roles)),default=str).encode()).hexdigest()
    report.update(status='ELIGIBLE',chosen_attempt=int(attempt),host_assignment_sha256=fingerprint,
                  binary_class_counts=audit_counts.to_dict(),prevalence={r:float(audit_counts.loc[r,'attack']/audit_counts.loc[r].sum()) for r in audit_counts.index},
                  partition_sizes={r:int(audit_counts.loc[r].sum()) for r in audit_counts.index},
                  caveat='Host/class confounding; host-disjoint is not verified physical device-disjoint.')
    f[['__host__','__binary__','__split__','__capture_file__','__row_id__']].to_csv(ROOT/'locked_binary_row_assignments.csv',index=False)
    pd.DataFrame([{'source_host':h,'role':['train','val','test'][g]} for h,g in sorted(assigned.items())]).to_csv(ROOT/'locked_binary_host_assignments.csv',index=False)
    audit_counts.to_csv(ROOT/'locked_binary_class_coverage.csv')
    return f,{r:np.flatnonzero(f['__split__'].eq(r).to_numpy()) for r in ('train','val','test')},report

D,IDX,audit=split_binary_hosts(RAW)
(ROOT/'binary_split_audit.json').write_text(json.dumps(audit,indent=2,default=str))
print(json.dumps(audit,indent=2,default=str))
if IDX is None:print('NOT EVALUABLE: binary host-disjoint split could not be qualified; no training will run')


## Model training and evaluation

All five training seeds use the **identical locked host partition**. Teacher logits remain aligned to training rows. Student baselines are compared using macro-F1, ECE, NLL and binary Brier score, with class counts and recall reported explicitly. Model selection uses validation accuracy only; there is no test-set-based hyperparameter selection.

In [ ]:
def seed_everything(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
KD_T,KD_ALPHA=3.,.7


In [ ]:
# 3 — Reused architectures; memory-safe train/inference
class EdgeMLP(nn.Module):
    def __init__(self,n_features,n_classes,hidden=(64,32)):
        super().__init__();layers=[]; d=n_features
        for h in hidden:layers +=[nn.Linear(d,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(.1)]; d=h
        self.net=nn.Sequential(*layers,nn.Linear(d,n_classes))
    def forward(self,x):return self.net(x)

class FTTransformer(nn.Module):
    def __init__(self,n_feat,n_classes,d=96,heads=8,layers=4):
        super().__init__();self.weight=nn.Parameter(torch.randn(n_feat,d)*.02)
        self.bias=nn.Parameter(torch.zeros(n_feat,d));self.cls=nn.Parameter(torch.randn(1,1,d)*.02)
        enc=nn.TransformerEncoderLayer(d_model=d,nhead=heads,dim_feedforward=2*d,dropout=.1,batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(enc,num_layers=layers)
        self.head=nn.Sequential(nn.LayerNorm(d),nn.ReLU(),nn.Linear(d,n_classes))
    def forward(self,x):
        t=x.unsqueeze(-1)*self.weight+self.bias
        t=torch.cat([self.cls.expand(x.size(0),-1,-1),t],1)
        return self.head(self.encoder(t)[:,0])

def train_nn(model,X,y,seed,epochs=EPOCHS,teacher_logits=None,alpha=KD_ALPHA,T=KD_T,ls=0.0,Xval=None,yval=None):
    seed_everything(seed);model=model.to(DEVICE)
    rng=torch.Generator().manual_seed(seed)
    arrays=[torch.from_numpy(X),torch.from_numpy(y)]
    if teacher_logits is not None:arrays.append(torch.from_numpy(np.asarray(teacher_logits,dtype=np.float32)))
    batch_size=64 if isinstance(model,FTTransformer) else (512 if DEVICE=='cpu' else 1024)
    dl=torch.utils.data.DataLoader(torch.utils.data.TensorDataset(*arrays),batch_size=batch_size,
                                   shuffle=True,generator=rng)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    best=(-np.inf,None)
    for ep in range(epochs):
        model.train()
        for b in dl:
            xx,yy=b[0].to(DEVICE),b[1].to(DEVICE)
            z=model(xx)
            loss=F.cross_entropy(z,yy,label_smoothing=ls)
            if teacher_logits is not None:
                zz=b[2].to(DEVICE)
                loss=(1-alpha)*loss+alpha*T*T*F.kl_div(F.log_softmax(z/T,dim=1),F.softmax(zz/T,dim=1),reduction='batchmean')
            opt.zero_grad();loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1);opt.step()
        score=np.mean(infer(model,Xval).argmax(1)==yval) if Xval is not None else -float(loss.detach().cpu())
        if score>best[0]:best=(score,{k:v.detach().cpu().clone() for k,v in model.state_dict().items()})
    model.load_state_dict(best[1]);return model

@torch.no_grad()
def infer(model,X,batch=2048):
    model.eval();res=[]
    for start in range(0,len(X),batch):res.append(model(torch.from_numpy(X[start:start+batch]).to(DEVICE)).cpu().numpy())
    return np.concatenate(res).astype(np.float32)

def fit_xgb(X,y,seed,k):
    try:import xgboost as xgb
    except ImportError:raise RuntimeError('pip install xgboost')
    clf=xgb.XGBClassifier(n_estimators=350,max_depth=6,learning_rate=.06,subsample=.8,colsample_bytree=.8,
                          tree_method='hist',n_jobs=4,random_state=seed,objective='multi:softprob' if k>2 else 'binary:logistic')
    clf.fit(X,y);return clf

def xgb_logits(m,X):return np.log(np.clip(m.predict_proba(X),1e-7,1))


In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, balanced_accuracy_score, recall_score
def ece(conf,correct,bins=15):
    which=np.clip(np.digitize(conf,np.linspace(0,1,bins+1)[1:-1]),0,bins-1)
    return float(sum(np.mean(which==b)*abs(np.mean(correct[which==b])-np.mean(conf[which==b])) for b in range(bins) if (which==b).any()))
def assess(z,y):
    p=softmax(z,axis=1);pr=p.argmax(axis=1);correct=(pr==y).astype(float)
    return {'accuracy':float(accuracy_score(y,pr)),'macro_f1':float(f1_score(y,pr,average='macro',zero_division=0)),
            'ece':ece(p.max(1),correct),'nll':float(-np.log(np.clip(p[np.arange(len(y)),y],1e-12,1)).mean())}



def assess_binary(z,y):
    m=assess(z,y);p=softmax(z,axis=1);pr=p.argmax(axis=1)
    m['brier']=float(np.mean((p[:,1]-y)**2))
    m['balanced_accuracy']=float(balanced_accuracy_score(y,pr))
    m['attack_recall']=float(recall_score(y,pr,pos_label=1,zero_division=0))
    m['benign_recall']=float(recall_score(y,pr,pos_label=0,zero_division=0))
    m['auroc']=float(roc_auc_score(y,p[:,1])) if np.unique(y).size==2 else np.nan
    tn,fp,fn,tp=confusion_matrix(y,pr,labels=[0,1]).ravel()
    m.update(tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))
    return m

def split_preprocess_binary(d,sp):
    mapping={'benign':0,'attack':1}
    y=d['__binary__'].map(mapping).to_numpy(dtype=np.int64)
    z=d.iloc[sp['train']][FEATURES].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    feats=[c for c in FEATURES if z[c].notna().any() and z[c].nunique(dropna=True)>1]
    if not feats:raise ValueError('No valid training-only numeric predictors')
    pipe=Pipeline([('imp',SimpleImputer(strategy='median')),('qt',QuantileTransformer(n_quantiles=min(1000,len(z)),output_distribution='normal',random_state=SPLIT_SEED))])
    pipe.fit(z[feats]);out={}
    for role,indices in sp.items():
        part=d.iloc[indices][feats].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
        out[role]=(pipe.transform(part).astype('float32'),y[indices])
    return out,feats


In [ ]:
RUN_SEEDS=[13,42,77,2024,31337]
columns=['seed','regime','protocol','n_test','accuracy','macro_f1','ece','nll','brier','balanced_accuracy','attack_recall','benign_recall','auroc','tn','fp','fn','tp']
records=[];errors=[]
if IDX is None:
    print('Training not attempted: ineligible locked partition')
else:
    prepared,feats=split_preprocess_binary(D,IDX)
    (xall,yall),(xv,yv),(xe,ye)=[prepared[role] for role in ('train','val','test')]
    # Fixed training subsample independent of model-seed. Validation/test remain complete.
    if len(yall)>MAX_TRAIN_ROWS:
        from sklearn.model_selection import train_test_split
        ix,_=train_test_split(np.arange(len(yall)),train_size=MAX_TRAIN_ROWS,random_state=SPLIT_SEED,stratify=yall)
        xt,yt=xall[ix],yall[ix]
    else:xt,yt=xall,yall
    train_support=dict(zip(*np.unique(yt,return_counts=True)))
    assert len(train_support)==2 and len(np.unique(yv))==2 and len(np.unique(ye))==2
    print('Training rows',len(xt),'validation',len(xv),'test',len(xe),'features',len(feats))
    pd.DataFrame({'feature':feats}).to_csv(ROOT/'binary_predictor_list.csv',index=False)
    for seed in RUN_SEEDS:
        try:
            seed_everything(seed);models={}
            models['scratch']=train_nn(EdgeMLP(len(feats),2),xt,yt,seed,Xval=xv,yval=yv)
            ft=train_nn(FTTransformer(len(feats),2),xt,yt,seed+200,Xval=xv,yval=yv)
            xgb=fit_xgb(xt,yt,seed,2)
            teachers={'ft':(infer(ft,xt),infer(ft,xv),infer(ft,xe)), 'xgb':(xgb_logits(xgb,xt),xgb_logits(xgb,xv),xgb_logits(xgb,xe))}
            staged=[];saved=[]
            for name,(trz,vaz,tez) in teachers.items():
                staged.append({'seed':seed,'regime':'teacher_'+name,'protocol':'locked_binary_source_host','n_test':len(ye),**assess_binary(tez,ye)})
                student=train_nn(EdgeMLP(len(feats),2),xt,yt,seed+300,teacher_logits=trz,Xval=xv,yval=yv)
                models['kd_'+name]=student
                saved.append((name,vaz,tez))
            for name,model in models.items():
                zz=infer(model,xe);vz=infer(model,xv);staged.append({'seed':seed,'regime':name,'protocol':'locked_binary_source_host','n_test':len(ye),**assess_binary(zz,ye)})
                saved.append((name,vz,zz))
            for name,vz,z in saved:
                np.savez_compressed(ROOT/f'binary_seed{seed}_{name}_validation.npz',logits=vz,labels=yv,
                                    row_ids=D.iloc[IDX['val']]['__row_id__'].to_numpy())
                np.savez_compressed(ROOT/f'binary_seed{seed}_{name}.npz',logits=z,labels=ye,
                                    row_ids=D.iloc[IDX['test']]['__row_id__'].to_numpy())
            records.extend(staged)
            pd.DataFrame(records,columns=columns).to_csv(ROOT/'binary_generalization_results.csv',index=False)
            print('COMPLETED seed',seed,'models',len(staged))
            del models,ft,xgb,teachers,saved;gc.collect()
            if DEVICE=='cuda':torch.cuda.empty_cache()
        except Exception as exc:
            errors.append({'seed':seed,'error':repr(exc)})
            print('FAILED',seed,repr(exc))
            pd.DataFrame(errors).to_csv(ROOT/'binary_training_errors.csv',index=False)
if not records:pd.DataFrame(columns=columns).to_csv(ROOT/'binary_generalization_results.csv',index=False)


## Paired comparisons and reviewer evidence

Paired effects are across **training seeds, not independent test sets**. We report seed-level differences without claiming independent deployment replications. Exact paired sign-flip tests with five seeds cannot attain two-sided p < 0.05; report effect sizes and uncertainty rather than calling them significant.

In [ ]:
from itertools import product
rr=pd.DataFrame(records,columns=columns)
if len(rr):
    summary=rr.groupby('regime',as_index=False).agg(seeds=('seed','nunique'),macro_f1_mean=('macro_f1','mean'),macro_f1_sd=('macro_f1','std'),ece_mean=('ece','mean'),ece_sd=('ece','std'),nll_mean=('nll','mean'),brier_mean=('brier','mean'),auroc_mean=('auroc','mean'))
else:summary=pd.DataFrame(columns=['regime','seeds','macro_f1_mean','ece_mean','nll_mean'])
summary.to_csv(ROOT/'binary_generalization_summary.csv',index=False)
pairs=[]
for regime in ('kd_ft','kd_xgb'):
    aa=rr[rr.regime=='scratch'].set_index('seed') if len(rr) else pd.DataFrame()
    bb=rr[rr.regime==regime].set_index('seed') if len(rr) else pd.DataFrame()
    if aa.empty or bb.empty:continue
    common=sorted(set(aa.index)&set(bb.index))
    for metric in ('macro_f1','ece','nll','brier','attack_recall'):
        delta=(bb.loc[common,metric]-aa.loc[common,metric]).to_numpy(float)
        p_exact=np.nan
        if len(delta):
            null=np.array([np.mean(np.array(sign)*delta) for sign in product((-1,1),repeat=len(delta))])
            p_exact=float(np.mean(np.abs(null)>=abs(np.mean(delta))-1e-12))
        pairs.append({'comparison':regime+' minus scratch','metric':metric,'paired_seeds':len(common),'mean_difference':float(np.mean(delta)) if len(delta) else np.nan,'sd_difference':float(np.std(delta,ddof=1)) if len(delta)>1 else np.nan,'exact_two_sided_signflip_p':p_exact})
pd.DataFrame(pairs,columns=['comparison','metric','paired_seeds','mean_difference','sd_difference','exact_two_sided_signflip_p']).to_csv(ROOT/'binary_paired_differences.csv',index=False)
manifest={'split_eligibility':audit.get('status'),'split_fingerprint':audit.get('host_assignment_sha256'),'completed_seeds':sorted(rr.seed.unique().tolist()) if len(rr) else [],'completed_model_evaluations':len(rr),'training_failures':errors,'task':'binary benign vs attack','physically_verified_devices':False,'chronological_or_cross_site_validation':False,'protocol_limitation':'single fixed source-host-proxy partition; host/attack confounding; distinct from 15-class task'}
(ROOT/'binary_completion_manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
print(summary.to_string(index=False));print('Manifest:',json.dumps(manifest,indent=2))


## Validation-selected operating threshold (R1.3, R2.1)
Select on validation only; preserve unseen-host test data for one evaluation. Probability calibration metrics are unchanged by threshold selection.


In [ ]:
# Reviewer R1.3 / R2.1 — validation-only fixed-threshold operating point
# IMPORTANT: This is decision-threshold optimization, NOT probability calibration.
# ECE, NLL and Brier score remain computed from the ORIGINAL unmodified probabilities.
# Choose a threshold once per model and seed from validation labels, never from test labels.
from sklearn.metrics import precision_score
from scipy.special import expit
from pathlib import Path
import re

# 0.99 is a pre-specified validation attack-recall constraint, not tuned to test labels.
MIN_VALIDATION_ATTACK_RECALL=0.99
THRESHOLDS=np.unique(np.r_[np.linspace(0.01,0.99,99),0.5])

def positive_scores(z):
    z=np.asarray(z,dtype=np.float64)
    if z.ndim!=2 or z.shape[1]!=2:raise ValueError('Expected aligned two-class logits')
    return expit(np.clip(z[:,1]-z[:,0],-80,80))

def selection_table(y,s,thresholds=THRESHOLDS):
    y=np.asarray(y,dtype=np.int64);s=np.asarray(s,dtype=np.float64)
    if len(y)!=len(s) or not set(np.unique(y)).issubset({0,1}) or len(np.unique(y))!=2:
        raise ValueError('Validation requires aligned scores and both classes')
    t=np.asarray(thresholds,dtype=np.float64)
    pred=(s[None,:]>=t[:,None]).astype(np.int8)
    attack=y==1;benign=~attack
    tp=np.sum(pred[:,attack]==1,axis=1);fn=attack.sum()-tp
    tn=np.sum(pred[:,benign]==0,axis=1);fp=benign.sum()-tn
    attack_recall=tp/np.maximum(tp+fn,1)
    benign_recall=tn/np.maximum(tn+fp,1)
    f1_attack=2*tp/np.maximum(2*tp+fp+fn,1)
    f1_benign=2*tn/np.maximum(2*tn+fp+fn,1)
    return pd.DataFrame({'threshold':t,'macro_f1':(f1_attack+f1_benign)/2,
                         'attack_recall':attack_recall,'benign_recall':benign_recall,
                         'fp':fp,'fn':fn,'tp':tp,'tn':tn})

def choose_threshold(y_val,p_val,recall_floor=MIN_VALIDATION_ATTACK_RECALL):
    table=selection_table(y_val,p_val)
    eligible=table.loc[table.attack_recall>=recall_floor-1e-12].copy()
    # Feasible operating point maximizes validation macro-F1; ties: closeness to 0.5.
    # If no threshold is feasible, revert to the fixed 0.5 baseline (explicitly flagged).
    if len(eligible):
        eligible['tie']=np.abs(eligible.threshold-0.5)
        best=eligible.sort_values(['macro_f1','tie','threshold'],ascending=[False,True,True],kind='stable').iloc[0]
        return float(best.threshold),'validation_feasible_max_f1',table
    return 0.5,'no_feasible_recall_floor_fallback_0.5',table

def threshold_metrics(y,p,t):
    y=np.asarray(y,dtype=np.int64);p=np.asarray(p,dtype=np.float64)
    pr=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pr,labels=[0,1]).ravel()
    return {'macro_f1':float(f1_score(y,pr,average='macro',zero_division=0)),
            'balanced_accuracy':float(balanced_accuracy_score(y,pr)),
            'attack_recall':float(tp/max(tp+fn,1)),
            'benign_recall':float(tn/max(tn+fp,1)),
            'false_positive_rate':float(fp/max(fp+tn,1)),
            'attack_precision':float(tp/max(tp+fp,1)),
            'tp':int(tp),'fp':int(fp),'tn':int(tn),'fn':int(fn)}

# Validation and test arrays must be saved by the patched training cell.
SEEDS=[13,42,77,2024,31337]
REGIMES=['scratch','kd_ft','kd_xgb','ft','xgb']
rows=[];checks=[];missing=[]
for seed in SEEDS:
    for regime in REGIMES:
        name=f'binary_seed{seed}_{regime}'
        vp=ROOT/f'{name}_validation.npz';tp=ROOT/f'{name}.npz'
        if not vp.exists() or not tp.exists():
            missing.append({'seed':seed,'regime':regime,'reason':'missing validation or test logits'})
            continue
        with np.load(vp) as vf,np.load(tp) as tf:
            zv,yv_local,rv=vf['logits'],vf['labels'],vf['row_ids']
            zt,yt_local,rt=tf['logits'],tf['labels'],tf['row_ids']
        assert len(set(rv))==len(rv) and len(set(rt))==len(rt)
        assert len(set(rv)&set(rt))==0, 'Validation/test row ID leakage'
        assert np.isfinite(zv).all() and np.isfinite(zt).all()
        sv,st=positive_scores(zv),positive_scores(zt)
        threshold,policy,table=choose_threshold(yv_local,sv)
        table.insert(0,'regime',regime);table.insert(0,'seed',seed)
        checks.append(table)
        for setting,t in [('default_0.5',0.5),('validation_selected',threshold)]:
            for split,y,p in [('validation',yv_local,sv),('test',yt_local,st)]:
                m=threshold_metrics(y,p,t)
                rows.append({'seed':seed,'regime':regime,'setting':setting,'split':split,
                             'threshold':t,'selection_policy':policy,'n':len(y),**m,
                             'auroc':float(roc_auc_score(y,p)),
                             'brier_unmodified':float(np.mean((p-y)**2)),
                             'nll_unmodified':float(-np.mean(y*np.log(np.clip(p,1e-12,1))+(1-y)*np.log(np.clip(1-p,1e-12,1)))),
                             'ece_original_argmax':ece(np.maximum(p,1-p),(np.where(p>=.5,1,0)==y).astype(float))})
results_threshold=pd.DataFrame(rows)
results_threshold.to_csv(ROOT/'threshold_results.csv',index=False)
if checks:pd.concat(checks,ignore_index=True).to_csv(ROOT/'validation_threshold_grid.csv',index=False)
pd.DataFrame(missing,columns=['seed','regime','reason']).to_csv(ROOT/'threshold_missing_files.csv',index=False)
summary_threshold=pd.DataFrame()
paired_threshold=pd.DataFrame()
if len(results_threshold):
    summary_threshold=results_threshold[results_threshold.split=='test'].groupby(['regime','setting'],as_index=False).agg(
        seeds=('seed','nunique'),threshold_mean=('threshold','mean'),macro_f1=('macro_f1','mean'),
        benign_recall=('benign_recall','mean'),attack_recall=('attack_recall','mean'),
        false_positive_rate=('false_positive_rate','mean'),nll=('nll_unmodified','mean'),ece=('ece_original_argmax','mean'))
    pairs=[]
    for model in REGIMES:
        part=results_threshold[(results_threshold.split=='test')&(results_threshold.regime==model)]
        baseline=part[part.setting=='default_0.5'].set_index('seed')
        tuned=part[part.setting=='validation_selected'].set_index('seed')
        common=sorted(set(baseline.index)&set(tuned.index))
        for metric in ('macro_f1','benign_recall','attack_recall','false_positive_rate'):
            diffs=(tuned.loc[common,metric]-baseline.loc[common,metric]).to_numpy()
            pairs.append({'regime':model,'metric':metric,'n_seeds':len(common),
                          'mean_tuned_minus_default':float(np.mean(diffs))})
    paired_threshold=pd.DataFrame(pairs)
summary_threshold.to_csv(ROOT/'threshold_summary.csv',index=False)
paired_threshold.to_csv(ROOT/'threshold_paired_effects.csv',index=False)
manifest_threshold={'method':'validation labels select operating threshold; test labels used only for final evaluation',
  'minimum_validation_attack_recall':MIN_VALIDATION_ATTACK_RECALL,
  'threshold_grid_count':int(len(THRESHOLDS)),
  'available_evaluations':int(len(results_threshold)),
  'missing_logits':missing,
  'protocol':'one unchanged source-host-disjoint binary partition; same split and preprocessing',
  'not_probability_calibration':True,
  'limitation':'selected threshold changes decisions, not probability calibration; no physical-device attribution; single host split'}
(ROOT/'threshold_manifest.json').write_text(json.dumps(manifest_threshold,indent=2))
print('Threshold result rows:',len(results_threshold),' missing pairs:',len(missing))
print(summary_threshold.to_string(index=False) if len(summary_threshold) else 'NO RESULTS')


## Controlled capture-feature ablation (R1.2, R1.3)
Risky transport counters and identifiers are removed only in the ablation arm. The preprocessing pipeline is fitted on training rows only. Teachers and students are retrained **from scratch** using exactly the locked host partition and same seed list. This provides a sensitivity analysis, not a proof of leakage.

In [ ]:
ABLATION_EXCLUDE={'tcp.seq','tcp.ack','udp.stream','tcp.nxtseq','tcp.analysis.acks_frame','tcp.analysis.ack_rtt', 'tcp.stream', 'tcp.seq_raw', 'tcp.ack_raw'}
AB_SEEDS=RUN_SEEDS
ab_records=[];ab_errors=[]
ab_columns=['seed','feature_set','regime','n_features','n_test','accuracy','macro_f1','ece','nll','brier','balanced_accuracy','attack_recall','benign_recall','auroc','tn','fp','fn','tp']
if IDX is None:
    print('ABRATION NOT EVALUABLE: locked split unavailable')
else:
    ab_features=[f for f in feats if f.lower() not in ABLATION_EXCLUDE]
    removed=sorted(set(feats)-set(ab_features))
    (ROOT/'ablation_feature_manifest.json').write_text(json.dumps({'baseline_features':feats,'safe_features':ab_features,'removed':removed,'locked_split':audit.get('host_assignment_sha256'),'preprocessing':'train-only refit','limitation':'Proxy host-shift, not physical-device generalization'},indent=2))
    print('Feature ablation:',len(feats),'->',len(ab_features),'removed:',removed)
    if not removed:
        print('No listed high-risk columns in this sampled common feature set; ablation is identical and is SKIPPED')
    else:
        old_features=FEATURES
        FEATURES=ab_features
        try:
            transformed,selected=split_preprocess_binary(D,IDX)
        finally:
            FEATURES=old_features
        (xa,ya),(xva,yva),(xea,yea)=[transformed[role] for role in ('train','val','test')]
        assert np.array_equal(ya,yall) and np.array_equal(yva,yv) and np.array_equal(yea,ye)
        if len(ya)>MAX_TRAIN_ROWS:
            from sklearn.model_selection import train_test_split
            ix,_=train_test_split(np.arange(len(ya)),train_size=MAX_TRAIN_ROWS,random_state=SPLIT_SEED,stratify=ya)
            xa,ya=xa[ix],ya[ix]
        for seed in AB_SEEDS:
            try:
                seed_everything(seed)
                scratch=train_nn(EdgeMLP(len(selected),2),xa,ya,seed,Xval=xva,yval=yva)
                ft=train_nn(FTTransformer(len(selected),2),xa,ya,seed+200,Xval=xva,yval=yva)
                xgb=fit_xgb(xa,ya,seed,2)
                teachers={'ft':(infer(ft,xa),infer(ft,xva),infer(ft,xea)),
                          'xgb':(xgb_logits(xgb,xa),xgb_logits(xgb,xva),xgb_logits(xgb,xea))}
                saved={'scratch':(infer(scratch,xva),infer(scratch,xea))}
                for name,(tr,vz,tz) in teachers.items():
                    saved['teacher_'+name]=(vz,tz)
                    student=train_nn(EdgeMLP(len(selected),2),xa,ya,seed+300,teacher_logits=tr,Xval=xva,yval=yva)
                    saved['kd_'+name]=(infer(student,xva),infer(student,xea))
                    del student
                batch=[]
                for name,(vz,tz) in saved.items():
                    batch.append({'seed':seed,'feature_set':'transport_counter_removed','regime':name,'n_features':len(selected),'n_test':len(yea),**assess_binary(tz,yea)})
                    np.savez_compressed(ROOT/f'ablation_seed{seed}_{name}_validation.npz',logits=vz,labels=yva,row_ids=D.iloc[IDX['val']]['__row_id__'].to_numpy())
                    np.savez_compressed(ROOT/f'ablation_seed{seed}_{name}.npz',logits=tz,labels=yea,row_ids=D.iloc[IDX['test']]['__row_id__'].to_numpy())
                ab_records.extend(batch)
                pd.DataFrame(ab_records,columns=ab_columns).to_csv(ROOT/'ablation_results.csv',index=False)
                print('ABLATION COMPLETE:',seed,len(batch))
                del scratch,ft,xgb,teachers,saved
                gc.collect()
                if DEVICE=='cuda':torch.cuda.empty_cache()
            except Exception as exc:
                ab_errors.append({'seed':seed,'error':repr(exc)})
                pd.DataFrame(ab_errors).to_csv(ROOT/'ablation_failures.csv',index=False)
                print('ABLATION FAILED',seed,repr(exc))
if not (ROOT/'ablation_results.csv').exists():pd.DataFrame(columns=ab_columns).to_csv(ROOT/'ablation_results.csv',index=False)


## Cross-arm paired analysis and confidence-geometry audit
The main analysis is baseline versus ablated predictors **within each regime**. Teacher/student geometries use aligned per-flow probabilities. The reported JS divergence is computed with clipping and renormalization and cannot be inferred from ECE alone.

In [ ]:
from scipy.special import softmax as scipy_softmax
from scipy.stats import spearmanr
from itertools import product
baseline_file=ROOT/'binary_generalization_results.csv'
abla=pd.read_csv(ROOT/'ablation_results.csv')
base=pd.read_csv(baseline_file) if baseline_file.exists() and baseline_file.stat().st_size>0 else pd.DataFrame()
if len(base):
    base=base.copy();base['feature_set']='all_eligible_features';base['n_features']=len(feats) if IDX is not None else np.nan
    combined=pd.concat([base,abla],ignore_index=True)
else:combined=abla.copy()
combined.to_csv(ROOT/'all_feature_arm_results.csv',index=False)
effects=[]
if len(base) and len(abla):
    for reg in sorted(set(base.regime)&set(abla.regime)):
        b=base[base.regime==reg].set_index('seed');a=abla[abla.regime==reg].set_index('seed')
        common=sorted(set(b.index)&set(a.index))
        for metric in ['macro_f1','ece','nll','benign_recall','attack_recall','auroc']:
            diffs=(a.loc[common,metric]-b.loc[common,metric]).to_numpy(dtype=float)
            if not len(diffs):continue
            permutations=np.array([np.mean(np.asarray(sign)*diffs) for sign in product([-1,1],repeat=len(diffs))]);p=float(np.mean(np.abs(permutations)>=abs(np.mean(diffs))-1e-12))
            effects.append({'regime':reg,'metric':metric,'n_seeds':len(diffs),'mean_ablation_minus_baseline':float(diffs.mean()),'sd':float(diffs.std(ddof=1)) if len(diffs)>1 else np.nan,'exact_signflip_p':p})
pd.DataFrame(effects,columns=['regime','metric','n_seeds','mean_ablation_minus_baseline','sd','exact_signflip_p']).to_csv(ROOT/'feature_ablation_paired.csv',index=False)
def safe_js(q,p):
    q=np.clip(np.asarray(q,dtype=float),1e-12,1);p=np.clip(np.asarray(p,dtype=float),1e-12,1)
    q=q/q.sum(axis=-1,keepdims=True);p=p/p.sum(axis=-1,keepdims=True)
    m=.5*(q+p)
    return .5*np.sum(q*np.log(q/m),axis=-1)+.5*np.sum(p*np.log(p/m),axis=-1)
geo=[]
for arm,prefix in [('all_eligible_features','binary'),('transport_counter_removed','ablation')]:
    for seed in RUN_SEEDS:
        for name in ['ft','xgb']:
            for split,suffix in [('validation','_validation.npz'),('test','.npz')]:
                teacher_tag=name if prefix=='binary' else 'teacher_'+name
                teacher=ROOT/f'{prefix}_seed{seed}_{teacher_tag}{suffix}';student=ROOT/f'{prefix}_seed{seed}_kd_{name}{suffix}'
                if not teacher.exists() or not student.exists():continue
                with np.load(teacher) as t,np.load(student) as s:
                    assert np.array_equal(t['row_ids'],s['row_ids']) and np.array_equal(t['labels'],s['labels'])
                    tp=scipy_softmax(t['logits'],axis=1);sp=scipy_softmax(s['logits'],axis=1)
                    tc=tp.max(1);sc=sp.max(1);rho=spearmanr(tc,sc).statistic
                    geo.append({'feature_set':arm,'seed':seed,'teacher':name,'split':split,'n':len(tc), 'js_mean':float(np.mean(safe_js(tp,sp))), 'confidence_mae':float(np.mean(np.abs(tc-sc))), 'confidence_spearman':float(rho) if np.isfinite(rho) else np.nan,'prediction_agreement':float(np.mean(tp.argmax(1)==sp.argmax(1)))})
pd.DataFrame(geo,columns=['feature_set','seed','teacher','split','n','js_mean','confidence_mae','confidence_spearman','prediction_agreement']).to_csv(ROOT/'host_geometry.csv',index=False)
print('Ablation effects',len(effects),'geometry comparisons',len(geo))


## External-validity feasibility gates and timing
Cross-dataset results require an independently checked feature mapping, not mere matching column names. Temporal evaluation requires validated capture provenance. The benchmark here measures the execution machine and **must not be labeled physical edge-hardware results**.

In [ ]:
# Independent cross-dataset mapping is a prerequisite; deliberately do not run a fake transfer test.
all_csv=list(INPUT.rglob('*.csv')) if INPUT.exists() else []
cic_count=sum('cic' in str(p).lower() or 'part-' in p.name.lower() for p in all_csv)
edge_count=sum('edge' in str(p).lower() for p in all_csv)
verified_mapping=INPUT/'verified_cross_dataset_feature_map.json'
checks={'ciciot_candidate_files':cic_count,'edge_candidate_files':edge_count,'verified_map_path':str(verified_mapping), 'mapping_exists':verified_mapping.exists(),'status':'NOT_EVALUABLE'}
if verified_mapping.exists():
    try:
        spec=json.loads(verified_mapping.read_text())
        items=spec.get('features',[])
        required={'source_feature','target_feature','unit','semantic_verification_reference'}
        valid=bool(items) and all(isinstance(x,dict) and required.issubset(x) and all(str(x[k]).strip() for k in required) for x in items)
        checks['mapped_features']=len(items);checks['status']='MAPPING_REQUIRES_MANUAL_REVIEW' if valid else 'INVALID_MAPPING'
    except Exception as exc:checks['mapping_error']=repr(exc);checks['status']='INVALID_MAPPING'
checks['cross_dataset_KD_run']=False
checks['explanation']='No cross-environment KD claim without audited semantic/unit mapping, original domain labels, train-only preprocessing and a truly independent evaluation.'
(ROOT/'cross_dataset_eligibility.json').write_text(json.dumps(checks,indent=2))
# Chronological test must not treat invalid timestamps as a valid session order.
time_status={'eligible':False,'reason':'Original Edge-IIoTset capture-time anomalies; no independently approved chronological capture metadata supplied','source_host_is_device_id':False}
(ROOT/'temporal_eligibility.json').write_text(json.dumps(time_status,indent=2))
# Device model latency: on current Kaggle machine, no claim of Raspberry Pi/device deployment.
import platform
bench=[]
if IDX is not None:
    # Use preprocessed single-flow feature vectors, do not change or read test labels during timing.
    sample=xt[:min(512,len(xt))].copy()
    m=EdgeMLP(len(feats),2).to(DEVICE).eval()
    if len(sample):
        with torch.no_grad():
            xx=torch.from_numpy(sample[:1]).to(DEVICE)
            for _ in range(25):_ = m(xx)
            if DEVICE=='cuda':torch.cuda.synchronize()
            for b in [1,256]:
                xx=torch.from_numpy(sample[:b]).to(DEVICE)
                start=time.perf_counter()
                with torch.no_grad():
                    for _ in range(200):_ = m(xx)
                if DEVICE=='cuda':torch.cuda.synchronize()
                duration=(time.perf_counter()-start)/200
                bench.append({'batch':len(xx),'total_batch_latency_ms':duration*1000,'per_flow_amortized_ms':duration*1000/len(xx),'model':'untrained EdgeMLP architecture proxy','includes_preprocessing':False,'platform':platform.platform(),'device':DEVICE})
pd.DataFrame(bench).to_csv(ROOT/'compute_only_latency_proxy.csv',index=False)
print('Gates:',checks['status'],'Timing rows',len(bench))


## Reviewer evidence and completion manifest
A completed notebook does **not** imply that physical gateway, chronology, device identities, or semantic cross-domain mapping exist. The manifest explicitly distinguishes completed tests from blocked tests.

In [ ]:
manifest={'task':'IEEE IoT-J R1 one-run evidence bundle','locked_partition':audit.get('host_assignment_sha256') if IDX is not None else None,'split_eligible':IDX is not None,'original_model_rows':len(base),'ablation_model_rows':len(abla),'ablation_failures':ab_errors,'geometry_rows':len(geo),'feature_ablation_tests':len(effects),'cross_dataset_status':checks['status'],'cross_dataset_KD_executed':False,'true_gateway_measured':False,'chronological_validated':False,'physical_devices_verified':False,'cross_dataset_feature_mapping_verified':False,'note':'Random seed repetitions reuse a single locked split; no new independent dataset. Kaggle compute latency is not on-device latency.'}
(ROOT/'all_in_one_manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
print(json.dumps(manifest,indent=2,default=str))
import zipfile
with zipfile.ZipFile(ROOT/'reviewer_evidence_tables.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.glob('*.csv'):z.write(p,p.name)
    for p in ROOT.glob('*.json'):z.write(p,p.name)
print('Saved',ROOT/'reviewer_evidence_tables.zip')
